# Pandas from First Principles
## Notebook 10: Reshaping Data

---

> **Series path:** Intro → Series → DataFrames → Indexing → Filtering → Grouping → Merging → String Operations → DateTime → **Reshaping ← You are here** → Reading & Writing Data

### What you will learn
| Topic | Key functions |
|---|---|
| Wide vs Long (tidy) format | conceptual |
| Pivot tables | `pivot_table()` |
| Wide → Long | `melt()` |
| Round-trip reshaping | `melt()` + `pivot_table()` |

**Prerequisite:** Notebooks 1–9 (especially GroupBy — Notebook 6)

In [ ]:
import pandas as pd
import numpy as np

print('pandas version:', pd.__version__)
print('Setup complete ✓')

---
## Part 1 — Wide Format vs Long (Tidy) Format

### What is it?
Real-world data almost always comes in one of two shapes:

- **Wide format** — each *category* gets its own column.
- **Long format** (also called *tidy* data) — each observation is one row; a separate column holds the category label.

### Intuition / Analogy
Imagine a **school report card**:

**Wide format** (one column per subject):
```
+-------+------+---------+---------+
| name  | math | science | english |
+-------+------+---------+---------+
| Riya  |  88  |   79    |   92    |
| Arjun |  74  |   82    |   68    |
+-------+------+---------+---------+
```

**Long format** (one row per student-subject pair):
```
+-------+---------+-------+
| name  | subject | score |
+-------+---------+-------+
| Riya  | math    |  88   |
| Riya  | science |  79   |
| Riya  | english |  92   |
| Arjun | math    |  74   |
| Arjun | science |  82   |
| Arjun | english |  68   |
+-------+---------+-------+
```

Same data — different shape.

### When is each format useful?

| Situation | Prefer |
|---|---|
| Human-readable reports / Excel | **Wide** |
| Plotting with seaborn / matplotlib | **Long** |
| GroupBy, aggregations, filtering | **Long** |
| Comparing specific columns side by side | **Wide** |
| ML feature matrices | **Wide** |

### Why reshaping matters
Many Pandas and visualisation operations **expect long format**. For example, `seaborn.barplot(x='subject', y='score', hue='name')` needs each subject to be a value in a column, not a separate column name.

Pandas gives us two power tools:
- `melt()` → Wide **to** Long
- `pivot_table()` → Long **to** Wide (with optional aggregation)

In [ ]:
# ── Datasets used throughout this notebook ────────────────────────────────────

# Long format — one row per (region, quarter) combination
sales_long = pd.DataFrame({
    'region':  ['North','North','North','South','South','South','East','East','East'],
    'quarter': ['Q1','Q2','Q3','Q1','Q2','Q3','Q1','Q2','Q3'],
    'revenue': [85000, 91000, 78000, 72000, 68000, 75000, 95000, 88000, 102000],
})

# Wide format — one row per student, one column per subject
student_scores = pd.DataFrame({
    'name':    ['Riya', 'Arjun', 'Priya', 'Karan'],
    'math':    [88, 74, 91, 62],
    'science': [79, 82, 88, 70],
    'english': [92, 68, 85, 75],
})

# Wide format — quarterly revenue per region
revenue_wide = pd.DataFrame({
    'region': ['North', 'South', 'East'],
    'Q1': [85000, 72000, 95000],
    'Q2': [91000, 68000, 88000],
    'Q3': [78000, 75000, 102000],
})

print('=== sales_long (Long format) ===')
print(sales_long)

print('\n=== student_scores (Wide format) ===')
print(student_scores)

print('\n=== revenue_wide (Wide format) ===')
print(revenue_wide)

---
## Part 2 — `pivot_table()`: Long → Wide with Aggregation

### What is it?
`pivot_table()` takes a **long-format** DataFrame and reshapes it into a **wide summary table** — similar to an Excel Pivot Table. Unlike a raw `pivot()`, it can **aggregate** duplicate entries.

### Why use it?
When you have repeated measurements (multiple sales in the same region + quarter), you need aggregation. `pivot_table()` handles that automatically.

### Syntax
```python
df.pivot_table(
    values    = 'column_to_aggregate',   # which numeric column to summarise
    index     = 'row_labels',            # unique values become row index
    columns   = 'col_labels',            # unique values become columns
    aggfunc   = 'mean',                  # how to aggregate duplicates
    fill_value= None,                    # replace NaN with this value
    margins   = False,                   # add row/column totals (Grand Total)
)
```

### Parameters explained
| Parameter | Type | What it does |
|---|---|---|
| `values` | str / list | The column(s) whose data you want to summarise |
| `index` | str / list | Column(s) whose values become the **row** index |
| `columns` | str / list | Column(s) whose values become the **column** headers |
| `aggfunc` | str / callable | How to combine duplicates: `'mean'`, `'sum'`, `'count'`, `'max'`, `'min'`, `np.median` |
| `fill_value` | scalar | Substitute for `NaN` in the result table |
| `margins` | bool | If `True`, adds an **All** row and column with totals |

### Return value
A new **DataFrame** with a MultiIndex or named columns derived from the `columns` parameter.

### Common mistake
> **Confusing `pivot()` with `pivot_table()`**
> - `pivot()` — no aggregation. Fails with a `ValueError` if there are duplicate (index, column) pairs.
> - `pivot_table()` — always aggregates. Safe to use when duplicates exist.
>
> **Rule of thumb:** Use `pivot_table()` by default. Only use `pivot()` when you are 100% sure every (row-key, col-key) pair is unique.

In [ ]:
# Basic pivot_table — mean revenue per region per quarter
pivot_mean = sales_long.pivot_table(
    values='revenue',
    index='region',
    columns='quarter',
    aggfunc='mean',   # default
)

print('pivot_table — mean revenue per region per quarter:')
print(pivot_mean)

In [ ]:
# aggfunc options: 'sum', 'count', 'max'
pivot_sum = sales_long.pivot_table(
    values='revenue',
    index='region',
    columns='quarter',
    aggfunc='sum',
)

print('aggfunc="sum" — total revenue per region per quarter:')
print(pivot_sum)

In [ ]:
# margins=True — adds row totals (All column) and column totals (All row)
pivot_with_margins = sales_long.pivot_table(
    values='revenue',
    index='region',
    columns='quarter',
    aggfunc='sum',
    margins=True,        # Grand Total row and column
)

print('pivot_table with margins=True (Grand Totals):')
print(pivot_with_margins)

In [ ]:
# fill_value — replacing NaN cells with 0
# Simulate a dataset with a missing combination
sales_sparse = pd.DataFrame({
    'region':  ['North', 'North', 'South', 'East', 'East'],
    'quarter': ['Q1',    'Q2',    'Q1',    'Q2',   'Q3'],
    'revenue': [85000,   91000,   72000,   88000,  102000],
})

# Without fill_value — NaN appears for missing combos
pivot_with_nan = sales_sparse.pivot_table(
    values='revenue', index='region', columns='quarter', aggfunc='sum'
)
print('Without fill_value (NaN for missing combos):')
print(pivot_with_nan)

# With fill_value=0 — NaN replaced by 0
pivot_filled = sales_sparse.pivot_table(
    values='revenue', index='region', columns='quarter',
    aggfunc='sum', fill_value=0
)
print('\nWith fill_value=0 (clean table):')
print(pivot_filled)

In [ ]:
# pivot() vs pivot_table() — side by side demo
print('=== pivot() — requires unique (index, columns) pairs ===')
try:
    # sales_long HAS duplicate (region, quarter) pairs → ValueError
    bad = sales_long.pivot(index='region', columns='quarter', values='revenue')
except ValueError as err:
    print(f'pivot() raised ValueError: {err}')

print('\n=== pivot_table() — handles duplicates by aggregating ===')
good = sales_long.pivot_table(
    index='region', columns='quarter', values='revenue', aggfunc='sum'
)
print(good)

# When pivot() IS safe — each (name, subject) pair is unique in student_scores
# First melt it to long format, then pivot
scores_long = student_scores.melt(id_vars='name', var_name='subject', value_name='score')
pivot_ok = scores_long.pivot(index='name', columns='subject', values='score')
print('\npivot() works fine when pairs are unique:')
print(pivot_ok)

---
## Part 3 — `melt()`: Wide → Long

### What is it?
`melt()` **unpivots** a wide-format DataFrame into a long format. It takes columns and "melts" them down into rows — perfect for transforming data before plotting or grouping.

### Intuition / Analogy
Think of a candle melting 🕯️. The wide table's multiple columns *melt down* into a single tall column of values, with a new label column tracking where each value came from.

```
WIDE (before melt)                  LONG (after melt)
+-------+------+---------+          +-------+---------+-------+
| name  | math | science |   →→→    | name  | subject | score |
+-------+------+---------+          +-------+---------+-------+
| Riya  |  88  |   79    |          | Riya  | math    |  88   |
| Arjun |  74  |   82    |          | Riya  | science |  79   |
+-------+------+---------+          | Arjun | math    |  74   |
                                    | Arjun | science |  82   |
                                    +-------+---------+-------+
```

### Syntax
```python
df.melt(
    id_vars    = ['col1', 'col2'],  # columns to keep as-is (identifier columns)
    value_vars = ['col3', 'col4'],  # columns to melt (default: all non-id_vars)
    var_name   = 'variable',        # name for the new label column
    value_name = 'value',           # name for the new values column
)
```

### Parameters explained
| Parameter | Type | What it does |
|---|---|---|
| `id_vars` | str / list | Columns that **stay fixed** — each appears in every melted row |
| `value_vars` | str / list | Columns to **melt**; omit to melt all non-id columns |
| `var_name` | str | Name for the new column that stores the old column names |
| `value_name` | str | Name for the new column that stores the old cell values |

### Return value
A new **long-format DataFrame** with `len(id_vars) + 2` columns and `rows × len(value_vars)` rows.

### When to use
When your **columns represent the same type of measurement** — e.g., months (Jan, Feb, Mar), quarters (Q1, Q2, Q3), or subjects (math, science, english). Those column names should instead be **values** in a category column.

### Common mistake
> **Not specifying `id_vars` correctly.**
> If you forget `id_vars`, the identifier column (like `'name'`) also gets melted into the values column, mixing strings with numbers — producing a column of `object` dtype where you expected numbers.

In [ ]:
# melt() — transform student_scores from wide to long
print('Original wide format:')
print(student_scores)

scores_long = student_scores.melt(
    id_vars='name',           # keep 'name' column as the identifier
    var_name='subject',       # old column names become values in 'subject'
    value_name='score',       # old cell values go into 'score'
)

print('\nAfter melt() — long format:')
print(scores_long)

In [ ]:
# Selective melt — only melt math and science, keep english wide
partial_melt = student_scores.melt(
    id_vars=['name', 'english'],    # keep 'name' and 'english' as-is
    value_vars=['math', 'science'], # only melt these two columns
    var_name='subject',
    value_name='score',
)

print('Partial melt — only math and science melted:')
print(partial_melt)

In [ ]:
# Demonstrate the common mistake — forgetting id_vars
bad_melt = student_scores.melt()  # no id_vars!

print('BAD melt — no id_vars (name column got melted too!):')
print(bad_melt.head(10))
print(f'\ndtype of value column: {bad_melt["value"].dtype}')  # object — mixed types!

---
## Part 4 — Practical Examples

Now let's put it all together with real-world style scenarios.

### Example A — Quarterly Sales: Wide → Long
The `revenue_wide` dataset has one column per quarter. For plotting or groupby operations, we want **one row per quarter**.

In [ ]:
# Example A: revenue_wide → long format using melt()
print('Original revenue_wide (wide format):')
print(revenue_wide)

revenue_long = revenue_wide.melt(
    id_vars='region',
    var_name='quarter',
    value_name='revenue',
)

print('\nAfter melt() — long format (one row per region-quarter):')
print(revenue_long.sort_values(['region', 'quarter']).reset_index(drop=True))

### Example B — Student Scores: Long → Wide (pivot_table)
After melting, we can use `pivot_table()` to reshape back into wide format — or to build a summary table showing average score per subject.

In [ ]:
# Example B: long format → pivot_table to get class average per subject
print('Long format (scores_long):')
print(scores_long)

class_avg = scores_long.pivot_table(
    values='score',
    index='name',
    columns='subject',
    aggfunc='mean',
)

print('\nPivot table — score per student per subject:')
print(class_avg)

### Example C — Round-Trip: Wide → melt → pivot_table → Wide
This confirms that `melt()` and `pivot_table()` are inverses of each other.

In [ ]:
# Example C: Round-trip — wide → melt → pivot_table → wide again
print('STEP 1 — Original wide format:')
print(revenue_wide)

# Step 2: melt to long
rev_long = revenue_wide.melt(
    id_vars='region',
    var_name='quarter',
    value_name='revenue',
)
print('\nSTEP 2 — After melt() (long):')
print(rev_long)

# Step 3: pivot_table back to wide
rev_back_to_wide = rev_long.pivot_table(
    values='revenue',
    index='region',
    columns='quarter',
    aggfunc='sum',
).reset_index()

# Remove the column name label 'quarter' from the columns axis
rev_back_to_wide.columns.name = None

print('\nSTEP 3 — After pivot_table() (wide again):')
print(rev_back_to_wide)

print('\nMatches original? →', revenue_wide.sort_values('region').reset_index(drop=True).equals(
    rev_back_to_wide.sort_values('region').reset_index(drop=True)
))

---
## Practice Questions

Use the three datasets defined earlier (`sales_long`, `student_scores`, `revenue_wide`).  
Try each question yourself before looking at the answer key.

| # | Difficulty | Topic |
|---|---|---|
| Q1 | 🟢 Easy | `pivot_table` — basic |
| Q2 | 🟢 Easy | `pivot_table` — margins |
| Q3 | 🟢 Easy | `melt` — basic |
| Q4 | 🟢 Easy | `melt` — revenue_wide |
| Q5 | 🟡 Medium | `melt` + `groupby` |
| Q6 | 🟡 Medium | `pivot_table` — count |
| Q7 | 🟡 Medium | Round-trip |
| Q8 | 🟡 Medium | `fill_value` |

---

**Q1 [Easy]** Use `pivot_table` on `sales_long`: rows = region, columns = quarter, values = revenue, aggfunc = sum.

In [ ]:
# Q1 — Your answer here


**Q2 [Easy]** Add `margins=True` to the Q1 result to see row and column totals.

In [ ]:
# Q2 — Your answer here


**Q3 [Easy]** Melt `student_scores` from wide to long format using `id_vars='name'`. Name the new columns `'subject'` and `'score'`.

In [ ]:
# Q3 — Your answer here


**Q4 [Easy]** Melt `revenue_wide` so each row is one `(region, quarter, revenue)` combination.

In [ ]:
# Q4 — Your answer here


**Q5 [Medium]** After melting `student_scores` (as in Q3), find the **average score per subject** across all students.

In [ ]:
# Q5 — Your answer here


**Q6 [Medium]** Use `pivot_table` with `aggfunc='count'` on `sales_long` to count how many entries exist per region per quarter.

In [ ]:
# Q6 — Your answer here


**Q7 [Medium]** Melt `revenue_wide` to long format, then use `pivot_table` to get back to the original wide shape. Reset the index and remove the column axis name.

In [ ]:
# Q7 — Your answer here


**Q8 [Medium]** Create a sparse dataset with missing region-quarter combinations, then use `pivot_table` with `fill_value=0` so no NaN appears in the result.

```python
sparse = pd.DataFrame({
    'region':  ['North', 'North', 'South', 'East'],
    'quarter': ['Q1',    'Q2',    'Q1',    'Q3'],
    'revenue': [85000,   91000,   72000,   102000],
})
```

In [ ]:
# Q8 — Your answer here


---
## ✅ Answer Key

In [ ]:
# ── Answer: Q1 ───────────────────────────────────────────────────────────────
print('Q1 — pivot_table: region × quarter, aggfunc=sum')
ans1 = sales_long.pivot_table(
    values='revenue',
    index='region',
    columns='quarter',
    aggfunc='sum',
)
print(ans1)

In [ ]:
# ── Answer: Q2 ───────────────────────────────────────────────────────────────
print('Q2 — pivot_table with margins=True')
ans2 = sales_long.pivot_table(
    values='revenue',
    index='region',
    columns='quarter',
    aggfunc='sum',
    margins=True,
)
print(ans2)

In [ ]:
# ── Answer: Q3 ───────────────────────────────────────────────────────────────
print('Q3 — melt student_scores to long format')
ans3 = student_scores.melt(
    id_vars='name',
    var_name='subject',
    value_name='score',
)
print(ans3)

In [ ]:
# ── Answer: Q4 ───────────────────────────────────────────────────────────────
print('Q4 — melt revenue_wide to (region, quarter, revenue)')
ans4 = revenue_wide.melt(
    id_vars='region',
    var_name='quarter',
    value_name='revenue',
)
print(ans4.sort_values(['region', 'quarter']).reset_index(drop=True))

In [ ]:
# ── Answer: Q5 ───────────────────────────────────────────────────────────────
print('Q5 — average score per subject after melting')
ans5_long = student_scores.melt(
    id_vars='name',
    var_name='subject',
    value_name='score',
)
ans5 = ans5_long.groupby('subject')['score'].mean().round(2)
print(ans5)

In [ ]:
# ── Answer: Q6 ───────────────────────────────────────────────────────────────
print('Q6 — pivot_table with aggfunc="count" (entries per region per quarter)')
ans6 = sales_long.pivot_table(
    values='revenue',
    index='region',
    columns='quarter',
    aggfunc='count',
    fill_value=0,
)
print(ans6)

In [ ]:
# ── Answer: Q7 ───────────────────────────────────────────────────────────────
print('Q7 — round-trip: revenue_wide → melt → pivot_table → wide')

# Step 1: melt
ans7_long = revenue_wide.melt(
    id_vars='region',
    var_name='quarter',
    value_name='revenue',
)

# Step 2: pivot_table back to wide
ans7 = ans7_long.pivot_table(
    values='revenue',
    index='region',
    columns='quarter',
    aggfunc='sum',
).reset_index()
ans7.columns.name = None  # remove axis label 'quarter'

print('Round-trip result:')
print(ans7)
print('\nOriginal revenue_wide:')
print(revenue_wide)

In [ ]:
# ── Answer: Q8 ───────────────────────────────────────────────────────────────
print('Q8 — fill_value=0 to replace NaN in sparse pivot table')

sparse = pd.DataFrame({
    'region':  ['North', 'North', 'South', 'East'],
    'quarter': ['Q1',    'Q2',    'Q1',    'Q3'],
    'revenue': [85000,   91000,   72000,   102000],
})

ans8 = sparse.pivot_table(
    values='revenue',
    index='region',
    columns='quarter',
    aggfunc='sum',
    fill_value=0,  # missing combos → 0 instead of NaN
)
print(ans8)

---
## ⚡ Quick Revision Table

| Concept | Function | Key Parameters | Use case |
|---|---|---|---|
| Long → Wide (with aggregation) | `pivot_table()` | `values`, `index`, `columns`, `aggfunc`, `fill_value`, `margins` | Summarise data by two categorical dimensions |
| Wide → Long | `melt()` | `id_vars`, `value_vars`, `var_name`, `value_name` | Prepare data for plots / groupby |
| Long → Wide (no aggregation) | `pivot()` | `index`, `columns`, `values` | Only when pairs are unique — no duplicates |
| Default aggfunc | `pivot_table()` | `aggfunc='mean'` | Average of all matching values |
| Grand totals | `pivot_table()` | `margins=True` | Adds an **All** row and column |
| Replace NaN | `pivot_table()` | `fill_value=0` | Avoid NaN in output table |
| Round-trip | `melt()` → `pivot_table()` | — | Confirm data integrity; shape transformation |

---
## 🎤 3 Interview Questions

**1. What is the difference between `pivot()` and `pivot_table()` in Pandas?**

> `pivot()` reshapes data **without aggregation** — it raises a `ValueError` if there are duplicate (index, column) pairs.  
> `pivot_table()` **aggregates duplicates** using a function (`mean`, `sum`, etc.) and is therefore safer and more general.  
> Use `pivot()` only when you are guaranteed each pair is unique; otherwise always prefer `pivot_table()`.

---

**2. What does `melt()` do, and when would you use it?**

> `melt()` converts a **wide-format** DataFrame (where categories are spread across columns) into a **long-format** DataFrame (where each observation is a single row).  
> You use it when column names represent values of the same variable — for example, months (Jan, Feb, Mar) or subjects (math, science, english) — and you need those values in a single column for use with `groupby`, seaborn, or scikit-learn.

---

**3. What is tidy data, and why does it matter for data analysis in Python?**

> Tidy (long) data follows three rules: (1) each variable is a column, (2) each observation is a row, (3) each type of observational unit is a table.  
> Most Pandas operations (`groupby`, `merge`, `apply`) and visualisation libraries (seaborn, Plotly) are designed to work with tidy data. Reshaping messy wide data into tidy long format is often a prerequisite step before any analysis.

---
## 🔜 What's Next?

**Notebook 11 — Reading & Writing Data**

We've been creating data by hand inside the notebook. In Notebook 11 we'll learn how to work with **real-world files**:

| Function | Purpose |
|---|---|
| `pd.read_csv()` | Read data from CSV files |
| `df.to_csv()` | Save a DataFrame to a CSV file |
| `pd.read_excel()` | Read from Excel (.xlsx) files |
| `pd.read_json()` | Parse JSON into a DataFrame |
| `pd.read_sql()` | Query a SQL database into a DataFrame |

You'll learn about encoding, separators, chunked reading for large files, and best practices for reproducible data pipelines.

---
*Pandas from First Principles — Notebook 10 of 12*